# AI-Based Resume Match Engine — benchmark in Google Colab

This notebook compares two ways of ranking resumes for a job:

* **TF-IDF** – keyword matching (how basic applicant-tracking systems work)
* **Sentence-Transformers** – a small Hugging Face transformer model (`all-MiniLM-L6-v2`, runs on PyTorch) that compares *meaning*, not just exact words

Data: the Kaggle **Resume Dataset** (~2,400 real resumes in 24 job categories).

**Before you start:** *Runtime → Change runtime type → T4 GPU*. On GPU the whole run takes a few minutes; on CPU it can take 10–15.

## 1. Upload the project zip
Upload `resume-match-engine.zip` (the file Claude sent you) when the button appears.

In [ ]:
from google.colab import files
import zipfile, os
uploaded = files.upload()
zip_name = next(iter(uploaded))
zipfile.ZipFile(zip_name).extractall(".")
os.chdir("resume-match-engine")
print("Working in", os.getcwd())

## 2. Install dependencies
Colab already has PyTorch; this adds the rest.

In [ ]:
%pip install -q -r requirements.txt

## 3. Download the Kaggle Resume Dataset
`kagglehub` downloads public Kaggle datasets. If it asks you to log in, either add your Kaggle API key
(Kaggle → Settings → *Create New Token*, then upload `kaggle.json`), or download `Resume.csv` from
https://www.kaggle.com/datasets/snehaanbhawal/resume-dataset and upload it with the file panel on the left.

In [ ]:
import kagglehub
data_path = kagglehub.dataset_download("snehaanbhawal/resume-dataset")
print("Dataset folder:", data_path)

In [ ]:
import pandas as pd
from benchmark.run_benchmark import load_resumes
df = load_resumes(data_path)
print(len(df), "resumes")
df["Category"].value_counts()

## 4. Run the benchmark
For each of the 24 categories there is a short job description in `data/category_jds.json`.
Every resume is ranked against every job; a resume counts as *correct* for a job if its category matches.

In [ ]:
!python benchmark/run_benchmark.py --data "$data_path" --out results

## 5. Results

In [ ]:
from IPython.display import Image, display
results = pd.read_csv("results/benchmark_results.csv", index_col=0)
display(results)
display(Image("results/benchmark_chart.png"))

**How to read it**

| Metric | Plain meaning |
|---|---|
| P@10 | Of the 10 resumes ranked highest for a job, how many are from the right category |
| MRR | How high the first correct resume appears (1.0 = always first) |
| nDCG@10 | Like P@10, but correct resumes at the very top count more |
| Top-1 acc | For each resume, is the best-matching job its true category? |

In [ ]:
# Where does each method do well or badly? (P@10 per category)
per_cat = pd.read_csv("results/per_category_p10.csv", index_col=0)
per_cat.sort_values("sbert" if "sbert" in per_cat else per_cat.columns[0])

## 6. Try it on your own resume
Paste your resume and a real job description below.

In [ ]:
from engine import ResumeMatcher
matcher = ResumeMatcher()   # loads all-MiniLM-L6-v2 from the Hugging Face Hub
my_resume = open("data/sample_resume.txt").read()   # or paste your resume text here
job = open("data/sample_jd.txt").read()             # or paste a job description here
r = matcher.match(my_resume, job)
print(f"Score {r.overall}/100 – {r.verdict}   (encoder: {r.backend})")
print("Missing required skills:", r.missing_required)
for tip in r.suggestions:
    print(" •", tip)

## 7. Save your results
Download the `results` folder (right-click in the file panel) and commit it to GitHub, or run the cell below.
Write the real numbers from the table above in your README and resume — not estimates.

In [ ]:
import shutil
shutil.make_archive("benchmark_results", "zip", "results")
files.download("benchmark_results.zip")